# Sales Data Extraction - Web Scraping

This notebook contains the first stage of the sales data pipeline: automated extraction of detailed sales reports from the company's sales platform.

The objective of this stage is to download daily Excel reports and organize them by month, creating the raw data source for later cleaning, validation and analysis.

> **Note:** Credentials and downloaded files are not included in this repository for security and privacy reasons.

## 1. Project context

The sales platform allows exporting detailed sales reports by product and document.  
However, the extraction must be performed by date range and the downloaded files need to be organized before being used in later stages of the pipeline.

This notebook automates the process using browser automation.

Main tasks:

1. Log into the sales platform.
2. Access the detailed sales report.
3. Apply date filters.
4. Export the daily Excel report.
5. Store the downloaded file in a structured folder.
6. Repeat the process for a defined date range.

## 2. Libraries and tools

This section imports the libraries required for browser automation, file handling and date management.

- **Selenium:** browser automation.
- **webdriver-manager:** automatic ChromeDriver installation.
- **datetime:** date range management.
- **pathlib / os / shutil:** folder and file handling.
- **time:** simple wait control during download polling.

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from webdriver_manager.chrome import ChromeDriverManager

from datetime import datetime, timedelta
from pathlib import Path
import time
import shutil
import os

## 3. Configuration

This section defines the main parameters used by the extraction process.

Credentials are loaded from environment variables instead of being written directly in the notebook.

Required environment variables:

- `FACTURAPY_USER`
- `FACTURAPY_PASSWORD`

This avoids exposing sensitive information in GitHub.

In [ ]:
# Base URLs
BASE_URL = "https://eval.facturapy.me"
LOGIN_URL = f"{BASE_URL}/login"
REPORT_URL = f"{BASE_URL}/reporte/detalle-venta"

# Credentials loaded from environment variables
USERNAME = os.getenv("FACTURAPY_USER")
PASSWORD = os.getenv("FACTURAPY_PASSWORD")

# Download folder
DOWNLOAD_DIR = Path.cwd() / "downloads"

# Date range
START_DATE = datetime(2026, 3, 1)
END_DATE = datetime(2026, 3, 2)

# Business rule
SKIP_SUNDAYS = True

# Create download folder if it does not exist
DOWNLOAD_DIR.mkdir(exist_ok=True)

## 4. Credential validation

Before starting the browser automation process, the notebook validates whether the required credentials are available.

This prevents the script from opening the browser and failing later during login.

In [ ]:
if USERNAME is None or PASSWORD is None:
    raise ValueError(
        "Missing credentials. Please set FACTURAPY_USER and FACTURAPY_PASSWORD as environment variables."
    )
else:
    print("[✓] Credentials loaded successfully")

## 5. Browser setup

This function creates and configures a Chrome browser session.

Main components used:

- `ChromeOptions`: defines browser preferences.
- `download.default_directory`: sets the automatic download folder.
- `Service`: starts ChromeDriver.
- `WebDriverWait`: allows explicit waits for dynamic web elements.

In [ ]:
def create_driver(download_dir: Path):
    """
    Creates and configures the Chrome WebDriver.

    Parameters
    ----------
    download_dir : Path
        Folder where downloaded files will be stored.

    Returns
    -------
    driver : selenium.webdriver.Chrome
        Browser automation driver.
    wait : selenium.webdriver.support.ui.WebDriverWait
        Explicit wait object for dynamic page elements.
    """

    chrome_options = webdriver.ChromeOptions()

    chrome_options.add_experimental_option("prefs", {
        "download.default_directory": str(download_dir),
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True
    })

    # Browser options
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")

    # Use this option only if the process is stable and does not require visual inspection
    # chrome_options.add_argument("--headless")

    service = Service(ChromeDriverManager().install())

    driver = webdriver.Chrome(
        service=service,
        options=chrome_options
    )

    wait = WebDriverWait(driver, 30)

    return driver, wait

## 6. Login function

This function logs into the sales platform.

Selenium tools used:

- `By.NAME`: locates input fields by their HTML name.
- `send_keys()`: enters user credentials.
- `Keys.RETURN`: submits the form.
- `WebDriverWait`: waits until the login process is completed.

In [ ]:
def login(driver, wait, username: str, password: str) -> None:
    """
    Logs into the sales platform.

    Parameters
    ----------
    driver : selenium.webdriver.Chrome
        Browser automation driver.
    wait : selenium.webdriver.support.ui.WebDriverWait
        Explicit wait object.
    username : str
        Platform username.
    password : str
        Platform password.
    """

    driver.get(LOGIN_URL)

    email_input = wait.until(
        EC.presence_of_element_located((By.NAME, "email"))
    )
    email_input.send_keys(username)

    password_input = driver.find_element(By.NAME, "password")
    password_input.send_keys(password)
    password_input.send_keys(Keys.RETURN)

    wait.until(lambda d: d.current_url != LOGIN_URL)

    print("[✓] Login completed")

## 7. Download control function

This helper function detects when a new Excel file appears in the download folder.

The function compares the files that existed before clicking the export button with the files available after the download starts.

In [ ]:
def wait_for_download(download_dir: Path, previous_files: set, timeout: int = 30) -> Path | None:
    """
    Waits until a new Excel file appears in the download folder.

    Parameters
    ----------
    download_dir : Path
        Folder where files are downloaded.
    previous_files : set
        Files that existed before starting the download.
    timeout : int
        Maximum waiting time in seconds.

    Returns
    -------
    Path | None
        Path of the downloaded Excel file, or None if no file is detected.
    """

    for _ in range(timeout):
        time.sleep(1)

        current_files = set(os.listdir(download_dir))
        new_files = current_files - previous_files

        for filename in new_files:
            if filename.endswith(".xlsx"):
                return download_dir / filename

    return None

## 8. Daily report extraction

This function downloads the detailed sales report for one specific date.

Process:

1. Open the report page.
2. Open the filter panel.
3. Set start and end date.
4. Search results.
5. Export the Excel file.
6. Move the file into a year-month folder.

In [ ]:
def download_daily_report(driver, wait, report_date: datetime, download_dir: Path) -> None:
    """
    Downloads the detailed sales report for a specific date.

    Parameters
    ----------
    driver : selenium.webdriver.Chrome
        Browser automation driver.
    wait : selenium.webdriver.support.ui.WebDriverWait
        Explicit wait object.
    report_date : datetime
        Date to be processed.
    download_dir : Path
        Folder where downloaded files will be stored.
    """

    date_str = report_date.strftime("%d-%m-%Y")

    try:
        print(f"Processing date: {date_str}")

        # Open report page
        driver.get(REPORT_URL)

        # Open filter panel
        filter_button = wait.until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "button.btn-filter"))
        )
        filter_button.click()

        # Locate date inputs
        start_date_input = wait.until(
            EC.element_to_be_clickable((By.ID, "fecha_emision_0"))
        )

        end_date_input = wait.until(
            EC.element_to_be_clickable((By.ID, "fecha_emision_1"))
        )

        # Set date values through JavaScript
        driver.execute_script(
            "arguments[0].value = arguments[1]",
            start_date_input,
            date_str
        )

        driver.execute_script(
            "arguments[0].value = arguments[1]",
            end_date_input,
            date_str
        )

        # Search filtered results
        search_button = wait.until(
            EC.element_to_be_clickable((By.ID, "ventaReportFilterButton"))
        )
        search_button.click()

        # Wait until table results are available
        wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "table tbody tr"))
        )

        # Register files before export
        files_before_download = set(os.listdir(download_dir))

        # Export report
        export_button = wait.until(
            EC.element_to_be_clickable((By.CLASS_NAME, "btn-export-dt"))
        )
        export_button.click()

        # Wait for downloaded file
        downloaded_file = wait_for_download(
            download_dir=download_dir,
            previous_files=files_before_download
        )

        if downloaded_file is None:
            print(f"[✗] No file downloaded for {date_str}")
            return

        # Create destination folder
        month_folder = report_date.strftime("%Y-%B")
        destination_folder = download_dir / month_folder
        destination_folder.mkdir(exist_ok=True)

        # Move and rename downloaded file
        destination_file = destination_folder / f"sales_detail_{date_str}.xlsx"

        shutil.move(downloaded_file, destination_file)

        print(f"[✓] File saved: {destination_file}")

    except Exception as error:
        print(f"[✗] Error processing {date_str}: {error}")

## 9. Date range execution

This function runs the extraction process across the selected date range.

Business rule:

- Sundays can be skipped because the company does not operate that day.

In [ ]:
def run_extraction(driver, wait, start_date: datetime, end_date: datetime) -> None:
    """
    Runs the extraction process across a date range.

    Parameters
    ----------
    driver : selenium.webdriver.Chrome
        Browser automation driver.
    wait : selenium.webdriver.support.ui.WebDriverWait
        Explicit wait object.
    start_date : datetime
        First date to process.
    end_date : datetime
        Last date to process.
    """

    current_date = start_date

    while current_date <= end_date:

        is_sunday = current_date.weekday() == 6

        if SKIP_SUNDAYS and is_sunday:
            print(f"Skipping Sunday: {current_date.strftime('%d-%m-%Y')}")
        else:
            download_daily_report(
                driver=driver,
                wait=wait,
                report_date=current_date,
                download_dir=DOWNLOAD_DIR
            )

        current_date += timedelta(days=1)

## 10. Main execution

This section executes the full extraction process.

The browser is closed inside the `finally` block to make sure the session ends even if an error occurs during extraction.

In [ ]:
driver, wait = create_driver(DOWNLOAD_DIR)

try:
    login(
        driver=driver,
        wait=wait,
        username=USERNAME,
        password=PASSWORD
    )

    run_extraction(
        driver=driver,
        wait=wait,
        start_date=START_DATE,
        end_date=END_DATE
    )

finally:
    driver.quit()
    print("[✓] Browser closed")

## 11. Output

At the end of the process, the downloaded files are stored using the following structure:

```text
downloads/
├── 2026-March/
│   ├── sales_detail_01-03-2026.xlsx
│   └── sales_detail_02-03-2026.xlsx
```

These raw files are used as input for the next stage of the pipeline: data consolidation, cleaning and validation.